In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    path for path in cwd_candidates
    if (path / "src" / "fpl_intelligence").exists() and (path / "analysis").exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from analysis.curated.player_gameweek_spine import build_player_gameweek_spine
from fpl_intelligence.config import DB_PATH

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

df = build_player_gameweek_spine(DB_PATH)
df.head()

In [ ]:
null_key_rows = df[df[["player_id", "gameweek"]].isnull().any(axis=1)].copy()
duplicate_rows = (
    df[df.duplicated(subset=["player_id", "gameweek"], keep=False)]
    .sort_values(["player_id", "gameweek"])
    .reset_index(drop=True)
)
misaligned_mask = df.apply(
    lambda row: not (
        len(row["fixture_ids"]) == len(row["opponent_team_ids"]) == len(row["was_home_flags"])
    ),
    axis=1,
)
misaligned_rows = df[misaligned_mask].copy()

validation_status = pd.DataFrame(
    [
        {"check": "null_player_id_or_gameweek", "failed_row_count": int(len(null_key_rows))},
        {"check": "duplicate_player_id_gameweek", "failed_row_count": int(len(duplicate_rows))},
        {"check": "misaligned_list_lengths", "failed_row_count": int(len(misaligned_rows))},
    ]
)
display(validation_status)

if not null_key_rows.empty:
    display(null_key_rows)
if not duplicate_rows.empty:
    display(duplicate_rows)
if not misaligned_rows.empty:
    display(misaligned_rows)

if not null_key_rows.empty or not duplicate_rows.empty or not misaligned_rows.empty:
    raise ValueError(
        "Spine validation failed: duplicate (player_id, gameweek), null player_id/gameweek, or misaligned list lengths detected."
    )

## 1. DATASET OVERVIEW

In [ ]:
overview_table = pd.DataFrame(
    [
        {"metric": "total_rows", "value": int(len(df))},
        {"metric": "unique_players", "value": int(df["player_id"].nunique())},
        {"metric": "unique_gameweeks", "value": int(df["gameweek"].nunique())},
    ]
)

rows_per_gameweek = (
    df.groupby("gameweek", dropna=False)
    .size()
    .rename("row_count")
    .reset_index()
    .sort_values("gameweek")
    .reset_index(drop=True)
)

rows_per_gameweek_distribution = (
    rows_per_gameweek["row_count"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99])
    .rename_axis("stat")
    .reset_index(name="value")
)

display(overview_table)
display(rows_per_gameweek)
display(rows_per_gameweek_distribution)

## 2. AGGREGATION SANITY CHECKS

In [ ]:
aggregation_metrics = ["minutes", "starts", "total_points"]
aggregation_summary = pd.DataFrame(
    [
        {
            "metric": metric,
            "min": df[metric].min(),
            "max": df[metric].max(),
            "mean": df[metric].mean(),
            "p0": df[metric].quantile(0.00),
            "p25": df[metric].quantile(0.25),
            "p50": df[metric].quantile(0.50),
            "p75": df[metric].quantile(0.75),
            "p95": df[metric].quantile(0.95),
            "p99": df[metric].quantile(0.99),
        }
        for metric in aggregation_metrics
    ]
)

minutes_gt_180 = (
    df.loc[df["minutes"] > 180]
    .sort_values(["minutes", "player_id", "gameweek"], ascending=[False, True, True])
    .reset_index(drop=True)
)
starts_gt_2 = (
    df.loc[df["starts"] > 2]
    .sort_values(["starts", "player_id", "gameweek"], ascending=[False, True, True])
    .reset_index(drop=True)
)
negative_total_points = (
    df.loc[df["total_points"] < 0]
    .sort_values(["total_points", "player_id", "gameweek"], ascending=[True, True, True])
    .reset_index(drop=True)
)

display(aggregation_summary)
display(minutes_gt_180)
display(starts_gt_2)
display(negative_total_points)

## 3. DGW STRUCTURE VALIDATION

In [ ]:
fixture_id_lengths = df["fixture_ids"].map(len)
fixture_length_distribution = (
    fixture_id_lengths.value_counts(dropna=False)
    .rename_axis("fixture_ids_length")
    .reset_index(name="row_count")
    .sort_values("fixture_ids_length")
    .reset_index(drop=True)
)

dgw_counts_table = pd.DataFrame(
    [
        {"check": "rows_with_length_0", "row_count": int((fixture_id_lengths == 0).sum())},
        {"check": "rows_with_length_1", "row_count": int((fixture_id_lengths == 1).sum())},
        {"check": "rows_with_length_2", "row_count": int((fixture_id_lengths == 2).sum())},
        {"check": "rows_with_length_gt_2", "row_count": int((fixture_id_lengths > 2).sum())},
    ]
)

rows_with_length_gt_2 = (
    df.loc[fixture_id_lengths > 2]
    .assign(fixture_ids_length=fixture_id_lengths[fixture_id_lengths > 2].values)
    .reset_index(drop=True)
)

display(fixture_length_distribution)
display(dgw_counts_table)
display(rows_with_length_gt_2)

## 4. JOIN INTEGRITY

In [ ]:
join_integrity_table = pd.DataFrame(
    [
        {
            "column": column,
            "null_count": int(df[column].isnull().sum()),
            "null_pct": df[column].isnull().mean() * 100,
        }
        for column in ["player_name", "element_type", "team_id", "opponent_team_ids"]
    ]
)

display(join_integrity_table)

## 5. ORDERING VALIDATION (CRITICAL)

In [ ]:
multi_fixture_rows = df.loc[df["fixture_ids"].map(len) > 1].copy()
ordering_sample_source = multi_fixture_rows.head(10).copy()

ordering_summary = ordering_sample_source.apply(
    lambda row: pd.Series(
        {
            "player_id": row["player_id"],
            "gameweek": row["gameweek"],
            "player_name": row["player_name"],
            "fixture_ids_length": len(row["fixture_ids"]),
            "opponent_team_ids_length": len(row["opponent_team_ids"]),
            "was_home_flags_length": len(row["was_home_flags"]),
            "list_lengths_equal": len(row["fixture_ids"]) == len(row["opponent_team_ids"]) == len(row["was_home_flags"]),
        }
    ),
    axis=1,
)

ordering_sample_rows = []
for _, row in ordering_sample_source.iterrows():
    for position, values in enumerate(
        zip(row["fixture_ids"], row["opponent_team_ids"], row["was_home_flags"]),
        start=1,
    ):
        fixture_id, opponent_team_id, was_home_flag = values
        ordering_sample_rows.append(
            {
                "player_id": row["player_id"],
                "gameweek": row["gameweek"],
                "player_name": row["player_name"],
                "position": position,
                "fixture_id": fixture_id,
                "opponent_team_id": opponent_team_id,
                "was_home_flag": was_home_flag,
            }
        )

ordering_sample_table = pd.DataFrame(
    ordering_sample_rows,
    columns=[
        "player_id",
        "gameweek",
        "player_name",
        "position",
        "fixture_id",
        "opponent_team_id",
        "was_home_flag",
    ],
)

display(ordering_summary)
display(ordering_sample_table)